In [ ]:
import torch

print(f'torch version : {torch.__version__}')
cuda = torch.cuda.is_available()
print(f'CUDA available: {cuda}')
if cuda:
    print(f'GPU           : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected — training on CPU will be very slow.')

In [ ]:
# torch, torchvision, opencv, scikit-learn are pre-installed on Kaggle
# ASSUMPTION: Kaggle base image includes torch>=2.0 and torchvision>=0.15
# albumentations powers the training-time augmentation in src/dataset.py
!pip install ultralytics timm albumentations --quiet

In [ ]:
# ── DATASET SETUP — MEDFISH101 ───────────────────────────────────────────
# MEDFISH101: 101 Mediterranean fish species, ~69k iNaturalist research-grade images
# (Front. Mar. Sci. 2026, doi:10.3389/fmars.2026.1754181).
#
# 1. Build the dataset locally:  python scripts/get_medfish101.py
#       (use --top-n N to download only the N most-populated species)
# 2. Zip it:  zip -r medfish101.zip data/medfish101/
# 3. Upload to Kaggle → Datasets → New Dataset → upload zip
# 4. Attach to this notebook: Add Data → Your Datasets → select it
#
# Expected layout (ImageFolder-compatible):
#   <dataset_name>/
#     Aidablennius_sphynx/  Apogon_imberbis/ ... Xyrichtys_novacula/
#
# INSTRUCTION: set DATA_ROOT to your attached MEDFISH101 dataset path.
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path

# TODO: replace with your Kaggle MEDFISH101 dataset slug/path once uploaded.
DATA_ROOT  = "/kaggle/input/medfish101"
OUTPUT_DIR = "/kaggle/working/outputs"

# TOP_N: 0 = train on all 101 species; set e.g. 30 to use only the N most-populated.
TOP_N = 0

# VERIFY: confirm the path exists and contains class subfolders before training
assert Path(DATA_ROOT).exists(), f'Dataset not found at {DATA_ROOT}'
classes_found = sorted(d.name for d in Path(DATA_ROOT).iterdir() if d.is_dir())
print(f'Species folders found: {len(classes_found)}')
print(classes_found[:5], '...')

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
import os

print(f'Contents of {DATA_ROOT} (first 10):')
print(sorted(os.listdir(DATA_ROOT))[:10])
print(f'\nTotal species folders: {len(os.listdir(DATA_ROOT))}')

In [ ]:
# Pull the real source (src/model.py, src/dataset.py, src/train.py) straight from the
# repo so this notebook always trains with the current code — including the Albumentations
# augmentation pipeline — instead of stale embedded copies.
import sys
from pathlib import Path

REPO_URL = "https://github.com/alvaropenuelas/real-time-fish-identification.git"
REPO_DIR = "/kaggle/working/real-time-fish-identification"

if not Path(REPO_DIR).exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

sys.path.insert(0, REPO_DIR)
print('src/ available from:', REPO_DIR)

In [ ]:
config = {
    'data_root'  : DATA_ROOT,
    'output_dir' : OUTPUT_DIR,
    'top_n'      : TOP_N,   # 0 = all species; N = N most-populated (MEDFISH101 subset knob)
    'epochs'     : 30,
    'batch_size' : 32,
    'num_workers': 2,   # ASSUMPTION: Kaggle GPU notebooks have >=2 CPU workers
    'lr'         : 1e-3,
    'patience'   : 5,
    'img_size'   : 224,
}

for k, v in config.items():
    print(f'  {k:<12}: {v}')

In [ ]:
import torch
from src.dataset import get_dataloaders
from src.model   import build_model
from src.train   import train

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

train_loader, val_loader, class_names = get_dataloaders(
    config['data_root'],
    batch_size=config['batch_size'],
    num_workers=config['num_workers'],
    top_n=config['top_n'],
)
print(f'Classes: {len(class_names)}  |  '
      f'train batches: {len(train_loader)}  |  val batches: {len(val_loader)}')

# num_classes is derived from the data — the deployed classifier (src/classifier.py)
# reads the class count back from the checkpoint, so no count is hardcoded anywhere.
model = build_model(num_classes=len(class_names))

best_acc = train(
    model, train_loader, val_loader,
    num_epochs=config['epochs'],
    output_dir=config['output_dir'],
    device=device,
    lr=config['lr'],
    patience=config['patience'],
)

import json
with open(f"{config['output_dir']}/classes.json", 'w') as f:
    json.dump(class_names, f, indent=2)

print(f'Training complete. Weights saved to {OUTPUT_DIR}/best_model.pt')

In [ ]:
from pathlib import Path

weights_path = Path(OUTPUT_DIR) / 'best_model.pt'
assert weights_path.exists(), f'ERROR: weights not found at {weights_path}'

size_mb = weights_path.stat().st_size / 1_048_576
print(f'best_model.pt : {size_mb:.1f} MB')

classes_path = Path(OUTPUT_DIR) / 'classes.json'
assert classes_path.exists(), 'ERROR: classes.json missing'
print(f'classes.json  : OK')

print('Ready to download.')

In [ ]:
# ── DOWNLOAD INSTRUCTIONS ────────────────────────────────────────────────
# 1. In the Kaggle notebook UI, click the 'Output' tab (right panel)
# 2. Navigate to outputs/best_model.pt and click the download icon
# 3. Also download outputs/classes.json
# 4. In your local repo:
#      mkdir -p weights
#      mv ~/Downloads/best_model.pt weights/model.pt
#      mv ~/Downloads/classes.json  outputs/classes.json
# 5. Verify the smoke test passes:
#      python -m unittest tests/test_classifier.py -v
# ─────────────────────────────────────────────────────────────────────────────
print('See cell comments above for download steps.')